# RAG 第 20 课：完整生产级 Pipeline + Ablation 消融实验

恭喜，你走到了整个 RAG 课程的收官。
前 19 课我们一块一块造了：

- chunking、embedding、BM25、hybrid、reranking
- evaluation、LLM-as-a-Judge
- query rewriting、multi-query、HyDE
- contextual compression
- conversational reformulation
- citations + grounding check

这一课做两件事：

1. **把它们串成一个真正的 production 级 pipeline**（可以作为你自己项目的起点）
2. **做 Ablation**：依次关掉某一环，看 Hit@1 / EM / F1 掉多少，回答一个工程师最该问的问题——**到底哪一步贡献最大？**

这比任何"最佳实践清单"都更有说服力。

In [ ]:
import gc
try:
    import torch
    if torch.cuda.is_available():
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print('GPU cleared.')
    else: print('CUDA not available.')
except Exception as e: print(f'no torch ({e})')

In [ ]:
import json, math, re
from collections import Counter, defaultdict
from typing import List
import numpy as np
from datasets import load_dataset
from openai import OpenAI

client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
preferred = ['qwen/qwen3.6-35b-a3b', 'qwen3.6-35b-a3b', 'qwen3.5-35b-a3b', 'qwen3.5-27b']
chat_model = next((m for m in preferred if m in model_ids), None)
embedding_model = next((m for m in model_ids if 'embed' in m.lower()), None)
print('chat =', chat_model, '| embed =', embedding_model)

## 1. 数据 + 基础检索组件（搬运）

In [ ]:
raw_ds = load_dataset('squad', split='validation[:160]')
documents=[]; seen={}; eval_rows=[]
for item in raw_ds:
    c = item['context'].strip()
    if c not in seen:
        seen[c] = len(documents)
        documents.append({'doc_id': len(documents), 'text': c, 'title': item['title']})
    did = seen[c]
    if item['answers']['text']:
        eval_rows.append({'question': item['question'].strip(), 'gold_doc_id': did,
                          'reference_answer': item['answers']['text'][0].strip()})

def tokenize(t): return re.findall(r'[a-zA-Z0-9]+', t.lower())
class BM25:
    def __init__(self,c,k1=1.5,b=0.75):
        self.k1,self.b=k1,b; self.dl=np.array([len(t) for t in c],dtype=np.float32)
        self.avgdl=float(self.dl.mean()); self.N=len(c); self.post=defaultdict(dict)
        for did,toks in enumerate(c):
            for tok,f in Counter(toks).items(): self.post[tok][did]=f
        self.idf={t:math.log(1+(self.N-len(p)+0.5)/(len(p)+0.5)) for t,p in self.post.items()}
    def score(self,qt):
        s=np.zeros(self.N,dtype=np.float32)
        for tok in qt:
            if tok not in self.post: continue
            idf=self.idf[tok]
            for did,f in self.post[tok].items():
                dl=self.dl[did]; s[did]+=idf*(f*(self.k1+1))/(f+self.k1*(1-self.b+self.b*dl/self.avgdl))
        return s
bm25=BM25([tokenize(d['text']) for d in documents])
def get_emb(t,bs=16):
    out=[]
    for i in range(0,len(t),bs):
        r=client.embeddings.create(model=embedding_model, input=t[i:i+bs])
        out.extend([np.array(x.embedding,dtype=np.float32) for x in r.data])
    return np.vstack(out)
def l2n(m): n=np.linalg.norm(m,axis=1,keepdims=True); return m/np.clip(n,1e-12,None)
doc_embeddings=l2n(get_emb([d['text'] for d in documents]))
print('docs =', len(documents), '| eval =', len(eval_rows))

## 2. 把每个能力封装成独立函数

这样做消融时，只要 `if config['use_xxx']` 就能切掉一层。

In [ ]:
def lexical_retrieve(q, top_k=20):
    s = bm25.score(tokenize(q))
    idx = np.argsort(s)[::-1][:top_k]
    return [(int(i), float(s[i])) for i in idx]
def dense_retrieve(text, top_k=20):
    qe = l2n(get_emb([text]))[0]; s = doc_embeddings @ qe
    idx = np.argsort(s)[::-1][:top_k]
    return [(int(i), float(s[i])) for i in idx]
def rrf(lst, k=60, top_k=20):
    f=defaultdict(float)
    for r in lst:
        for rk,(d,_) in enumerate(r,1): f[d]+=1/(k+rk)
    return [(d,s) for d,s in sorted(f.items(), key=lambda x:x[1], reverse=True)[:top_k]]

def rewrite_query(q):
    sys='Rewrite into a cleaner search query. Preserve intent and entities. Do not add facts. Output only the query.'
    r=client.chat.completions.create(model=chat_model,temperature=0,
        messages=[{'role':'system','content':sys},{'role':'user','content':q}])
    return r.choices[0].message.content.strip().strip('"')

def llm_rerank(q, cand, top_n):
    docs='\n\n'.join([f'DOC_{i+1} doc_id={c["doc_id"]}\n{c["text"][:400]}' for i,c in enumerate(cand)])
    sys='Rerank docs by usefulness for the question. Return strict JSON {"ranked_doc_ids":[...]}.'
    usr=f'Question: {q}\n\nCandidates:\n{docs}'
    r=client.chat.completions.create(model=chat_model,temperature=0,
        messages=[{'role':'system','content':sys},{'role':'user','content':usr}])
    raw=r.choices[0].message.content.strip()
    m=re.search(r'\{.*\}',raw,flags=re.S)
    if not m: return cand[:top_n]
    try:
        ids=json.loads(m.group(0)).get('ranked_doc_ids',[])
        mapp={c['doc_id']:c for c in cand}
        out=[mapp[i] for i in ids if i in mapp]
        seen={c['doc_id'] for c in out}
        for c in cand:
            if c['doc_id'] not in seen: out.append(c)
        return out[:top_n]
    except Exception: return cand[:top_n]

def compress(q, text):
    sys='Extract only sentences from the passage that help answer the question. Copy verbatim. If nothing relevant, output NONE.'
    usr=f'Question: {q}\n\nPassage:\n{text}\n\nRelevant sentences:'
    r=client.chat.completions.create(model=chat_model,temperature=0,
        messages=[{'role':'system','content':sys},{'role':'user','content':usr}])
    out=r.choices[0].message.content.strip()
    return '' if out.upper().startswith('NONE') else out

## 3. 完整 pipeline：config-driven

一个 config 控制每一层开关。这就是生产里 ablation 和 A/B 测试的常见写法。

In [ ]:
def run_pipeline(question, config):
    # step 1: query rewriting
    q = rewrite_query(question) if config.get('use_rewrite') else question

    # step 2: retrieval (hybrid 或 dense-only)
    cand_k = config.get('cand_k', 20)
    if config.get('use_hybrid', True):
        cand = rrf([lexical_retrieve(q, cand_k), dense_retrieve(q, cand_k)], top_k=cand_k)
    else:
        cand = dense_retrieve(q, top_k=cand_k)
    cand_rows = [{'doc_id': d, 'text': documents[d]['text'], 'title': documents[d]['title']} for d,_ in cand]

    # step 3: rerank
    top_n = config.get('top_n', 3)
    if config.get('use_rerank'):
        top_docs = llm_rerank(question, cand_rows, top_n)
    else:
        top_docs = cand_rows[:top_n]

    # step 4: compression
    if config.get('use_compression'):
        ctx_parts=[]
        for i,c in enumerate(top_docs,1):
            snip = compress(question, c['text'])
            if snip: ctx_parts.append(f'[Document {i}] {snip}')
        ctx = '\n\n'.join(ctx_parts) if ctx_parts else '(no relevant snippets)'
    else:
        ctx = '\n\n'.join([f'[Document {i}] title: {c["title"]}\n{c["text"]}' for i,c in enumerate(top_docs,1)])

    # step 5: answer (optionally with citations)
    if config.get('use_citations'):
        sys='Answer using only the documents. Return strict JSON {"sentences":[{"text":..., "citations":[doc_id,...]}]}.'
        usr=f'Question: {question}\n\nDocuments:\n' + '\n\n'.join([f'[doc_id={c["doc_id"]}]\n{c["text"]}' for c in top_docs]) + '\n\nReturn only JSON.'
        r=client.chat.completions.create(model=chat_model,temperature=0,
            messages=[{'role':'system','content':sys},{'role':'user','content':usr}])
        raw=r.choices[0].message.content.strip()
        m=re.search(r'\{.*\}',raw,flags=re.S)
        try:
            data=json.loads(m.group(0)) if m else {'sentences':[]}
            ans=' '.join([s.get('text','') for s in data.get('sentences',[])]).strip()
        except Exception: ans=raw
    else:
        sys='Answer only from the context. If not supported, reply NOT_FOUND. Keep short.'
        usr=f'Question: {question}\n\nContext:\n{ctx}\n\nReturn only the answer.'
        r=client.chat.completions.create(model=chat_model,temperature=0,
            messages=[{'role':'system','content':sys},{'role':'user','content':usr}])
        ans=r.choices[0].message.content.strip()

    return {'query_used': q, 'top_doc_ids': [c['doc_id'] for c in top_docs], 'answer': ans}

## 4. Ablation：依次开关某一层

用 `FULL` 作为最强 baseline，然后逐个关掉某一层，看哪一层最关键。

In [ ]:
FULL = {
    'use_rewrite': True,
    'use_hybrid': True,
    'use_rerank': True,
    'use_compression': True,
    'use_citations': False,  # 让答案格式和其他配置对齐，便于 EM/F1 比较
    'cand_k': 15, 'top_n': 3,
}

ABLATIONS = {
    'FULL': FULL,
    '-rewrite': {**FULL, 'use_rewrite': False},
    '-hybrid (dense only)': {**FULL, 'use_hybrid': False},
    '-rerank': {**FULL, 'use_rerank': False},
    '-compression': {**FULL, 'use_compression': False},
    'MIN (dense only, no rewrite/rerank/compress)': {'use_rewrite': False,'use_hybrid': False,'use_rerank': False,'use_compression': False,'cand_k': 15,'top_n': 3},
}

In [ ]:
def norm(t):
    return ' '.join([re.sub(r'[^a-z0-9]+','',x.lower()) for x in t.split() if re.sub(r'[^a-z0-9]+','',x.lower())])
def em(p,r): return 1.0 if norm(p)==norm(r) else 0.0
def f1(p,r):
    pt=norm(p).split(); rt=norm(r).split()
    if not pt or not rt: return 0.0
    ov=sum((Counter(pt)&Counter(rt)).values())
    if ov==0: return 0.0
    pr,rc=ov/len(pt),ov/len(rt); return 2*pr*rc/(pr+rc)

eval_subset = eval_rows[:5]
table = {}
for name, cfg in ABLATIONS.items():
    print(f'\n>>> running ablation: {name}')
    scores = {'hit@1': [], 'em': [], 'f1': []}
    for i, row in enumerate(eval_subset, 1):
        print(f'  {i}/{len(eval_subset)}: {row["question"]}')
        try:
            out = run_pipeline(row['question'], cfg)
            hit1 = 1.0 if out['top_doc_ids'] and out['top_doc_ids'][0]==row['gold_doc_id'] else 0.0
            scores['hit@1'].append(hit1)
            scores['em'].append(em(out['answer'], row['reference_answer']))
            scores['f1'].append(f1(out['answer'], row['reference_answer']))
        except Exception as e:
            print('    error:', e)
            scores['hit@1'].append(0.0); scores['em'].append(0.0); scores['f1'].append(0.0)
    table[name] = {k: float(np.mean(v)) for k,v in scores.items()}

print('\n=== Ablation summary ===')
print(f'{"config":50s} {"Hit@1":>8s} {"EM":>8s} {"F1":>8s}')
for name, sc in table.items():
    print(f'{name:50s} {sc["hit@1"]:>8.3f} {sc["em"]:>8.3f} {sc["f1"]:>8.3f}')

## 5. 消融结果怎么读

一个常见的判读方式：

- 从 `FULL` 拿掉某一层后，指标掉得越多 → 那一层**对这份数据最有用**
- 如果拿掉后几乎不掉 → 那一层**在这份数据上是冗余的**，可以省成本
- 如果拿掉后反而**变好** → 那一层的 prompt 或实现有问题（这在真实项目里非常常见！）

这就是为什么要做消融：**不做消融，你就只是在堆组件；做完消融，你才知道哪些组件值得保留。**

## 6. 全课程回顾

回顾 20 课的脉络：

- **1–10**：toy 实现讲清楚每个概念的"味道"
- **11**：把检索和 LLM 换成真实版本
- **12**：真实 reranking
- **13**：LLM-as-a-Judge 评估
- **14**：真实 hybrid（BM25 + dense + RRF）
- **15**：Query Rewriting / Multi-Query
- **16**：HyDE
- **17**：Contextual Compression
- **18**：Conversational RAG
- **19**：Citations & Grounding
- **20**：生产 Pipeline + Ablation

到这里，你已经有了一个**可以直接改造成产品**的 RAG 骨架。

下一步的方向由你自己的数据决定：
- 文档特别长 → 研究 chunking 策略 + small-to-big retrieval
- 多语言 → 多语言 embedding + BM25 分词
- 结构化数据混杂 → Text-to-SQL / Text-to-API 与 RAG 的结合
- 延迟敏感 → embedding/向量库工程化（FAISS/Milvus/pgvector）+ 缓存

祝你在自己的项目里玩得开心。